# DM4CT extended diffusion baselines for sparse-view CT

This notebook preserves the supplied DiffPDHG experiment as an optional, pinned reference path and runs four pixel-diffusion baselines through one DM4CT/AAPM/ASTRA pipeline:

`dps`, `red_diff`, `dmplug`, and `dds`.

The default `RUN_MODE="smoke"` uses the normalized AAPM example slice shipped at the pinned DM4CT commit. `validation` requires a disjoint validation directory. `full` requires the fixed 100 normalized AAPM test TIFFs configured below.


## Experiment contract

**Default paper protocol**

| Item | Exact value |
|---|---|
| Dataset | 2016 AAPM Low Dose CT Grand Challenge; DM4CT preprocessing |
| Full test slices | 100 fixed axial slices: `L506_000` through `L506_099` |
| Smoke slice | `L506_000` from the pinned DM4CT repository |
| Validation slices | `L333_000` through `L333_004`, disjoint from patient L506 |
| Image dimensions | `1 x 512 x 512` |
| Image normalization | global AAPM min/max mapped to `[-1, 1]`; metrics clip to `[-1, 1]`, `data_range=2`, no FOV mask |
| Diffusion checkpoint | `jiayangshi/lodochallenge_pixel_diffusion` at revision `b7e291c2febb28016a1ba1d639704822eb0d0793` |
| Projection angles | 80 values from DM4CT's `np.linspace(0, pi, 80)` convention |
| Detector bins | 512 |
| CT geometry | ASTRA `parallel3d`, one detector row, unit detector spacing, one 512 x 512 slice |
| Photon model | `I0=5000`, target average transmittance `0.5` |
| Cached observations | raw counts, normalized transmission, and DM4CT negative-log line integrals rescaled by the per-image attenuation factor |
| Random seed | `BASE_SEED=99`; measurement seed is `99 + image_index` |
| GPU | CUDA required; A100 preferred; the actual model is printed and saved at runtime |
| Pinned packages | diffusers 0.32.2, huggingface-hub 0.28.0, transformers 4.48.3, scikit-image 0.25.2, tifffile 2025.2.18, pandas 2.2.3, LPIPS 0.1.4, ASTRA 2.5.0 |
| Repositories | DM4CT `49b3e5907178b56338d55c17944de17168a7a0a1`; supplied DiffPDHG branch `35db5260db51809e789e913ea481b069bada0ac9` |

PyTorch and its CUDA libraries are supplied by the Colab GPU runtime rather than reinstalled; their exact versions are asserted, printed, and stored in `environment_manifest.json`.

**Difference from the supplied notebook.** The supplied notebook defaults to one of three small L067 demo TIFFs, a custom non-ASTRA differentiable projector, raw Poisson counts, `I0=10000`, and seed 99. It contains no DPS or RED-diff cells. This notebook therefore makes the requested paper protocol an explicit new default and keeps the original DiffPDHG launcher optional and separate instead of silently changing its historical results.


## 2. Runtime and GPU check

Select a GPU runtime in Colab. A CUDA GPU is a hard requirement because both ASTRA's direct 3D projector and the 512 x 512 diffusion model run on CUDA.


In [ ]:
import platform
import subprocess
import sys

import torch

print(subprocess.run(["nvidia-smi"], check=True, text=True, capture_output=True).stdout)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required. In Colab select Runtime > Change runtime type > GPU.")
if tuple(int(part) for part in torch.__version__.split("+")[0].split(".")[:2]) < (2, 5):
    raise RuntimeError(f"PyTorch >=2.5 is required; found {torch.__version__}")
print(
    {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "torch_cuda": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0),
    }
)


## 3. Repository checkout and pinned environment installation

DM4CT is checked out at a commit that contains all four requested pipelines, including DDS. The supplied DiffPDHG repository is also pinned so its optional reference launcher and commit remain reproducible. ASTRA 2.5.0 is the sole intentional update from DM4CT's 2.3.0 environment because 2.5.0 provides current Colab/PyPI CUDA wheels while retaining the direct projector API used here.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

DM4CT_URL = "https://github.com/DM4CT/DM4CT.git"
DM4CT_COMMIT = "49b3e5907178b56338d55c17944de17168a7a0a1"
DIFFPDHG_URL = "https://github.com/Seif-Hussein/dyscode.git"
DIFFPDHG_COMMIT = "35db5260db51809e789e913ea481b069bada0ac9"
CHECKPOINT_REVISION = "b7e291c2febb28016a1ba1d639704822eb0d0793"

DM4CT_DIR = Path("/content/DM4CT")
DIFFPDHG_DIR = Path("/content/dyscode_reference")

def checkout_pinned(url, directory, commit):
    directory = Path(directory)
    if not directory.exists():
        subprocess.run(
            ["git", "clone", "--filter=blob:none", "--no-checkout", url, str(directory)],
            check=True,
        )
    if not (directory / ".git").exists():
        raise RuntimeError(f"Existing path is not a git checkout: {directory}")
    subprocess.run(["git", "-C", str(directory), "fetch", "--depth", "1", "origin", commit], check=True)
    subprocess.run(["git", "-C", str(directory), "checkout", "--detach", commit], check=True)
    actual = subprocess.check_output(
        ["git", "-C", str(directory), "rev-parse", "HEAD"], text=True
    ).strip()
    if actual != commit:
        raise RuntimeError(f"Commit mismatch for {directory}: {actual} != {commit}")
    return actual

checkout_pinned(DM4CT_URL, DM4CT_DIR, DM4CT_COMMIT)
checkout_pinned(DIFFPDHG_URL, DIFFPDHG_DIR, DIFFPDHG_COMMIT)

PIP_PACKAGES = [
    "astra-toolbox==2.5.0",
    "diffusers==0.32.2",
    "huggingface-hub==0.28.0",
    "transformers==4.48.3",
    "scikit-image==0.25.2",
    "tifffile==2025.2.18",
    "pandas==2.2.3",
    "lpips==0.1.4",
    "accelerate==1.2.1",
]
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *PIP_PACKAGES], check=True)
print("Pinned repositories and Python packages are ready.")


## 4. Imports and reproducibility

Every measurement and method resets Python, NumPy, CPU PyTorch, and CUDA PyTorch seeds. cuDNN deterministic mode is enabled. ASTRA CUDA forward/backprojection can still use GPU accumulation orders that are not guaranteed bitwise deterministic across GPU models or driver versions; the manifest records both.


In [ ]:
import contextlib
import csv
import gc
import hashlib
import importlib.metadata
import json
import logging
import math
import os
import platform
import random
import shutil
import sys
import time
import traceback
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional

import astra
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from diffusers import DDIMScheduler, DDPMPipeline, DDPMScheduler
from diffusers.pipelines.pipeline_utils import ImagePipelineOutput
from diffusers.utils.torch_utils import randn_tensor
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from tifffile import imread, imwrite

sys.path.insert(0, str(DM4CT_DIR))
from condition_methods import DDS, DMPlug, PosteriorSampling, RedDiff
from ct_reconstruction import fbp, sirt
from forward_operators_ct import NoNoise, Operator
from pipelines import (
    DDPMPipelineDDS,
    DDPMPipelineDMPlug,
    DDPMPipelineDPS,
    DDPMPipelineRedDiff,
)

DEVICE = torch.device("cuda")

def set_all_seeds(seed):
    seed = int(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False
    if hasattr(torch.backends.cudnn, "allow_tf32"):
        torch.backends.cudnn.allow_tf32 = False

set_all_seeds(99)


## 5. Central configuration

Edit this cell only. Full-mode hyperparameters are frozen official DM4CT starting points. The validation cell explores no more than three one-factor variants for each new method and never mutates the full-test configuration.


In [ ]:
RUN_MODE = "smoke"  # "smoke" | "validation" | "full"
METHODS = [
    "dps",
    "red_diff",
    "dmplug",
    "dds",
]

DATA_ROOT = "/content/drive/MyDrive/dm4ct_data/lodochallenge/test"
VALIDATION_ROOT = "/content/drive/MyDrive/dm4ct_data/lodochallenge/validation"
OUTPUT_ROOT = "/content/dm4ct_extended_outputs"
MODEL_PATH_OR_HF_ID = "jiayangshi/lodochallenge_pixel_diffusion"
NUM_ANGLES = 80
NUM_DETECTORS = 512
I0 = 5000.0
TRANSMITTANCE = 0.5
BASE_SEED = 99
SAVE_RECONSTRUCTIONS = True
RESUME = True
COMPUTE_LPIPS = False
ENABLE_POISSON_DMPLUG = False
ENABLE_LINEARIZED_TRACK = True
IMPLEMENTATION_REVISION = "dm4ct-extended-2026-07-24-v1"

SMOKE_IMAGE_IDS = ["L506_000"]
VALIDATION_IMAGE_IDS = [f"L333_{index:03d}" for index in range(5)]
FULL_IMAGE_IDS = [f"L506_{index:03d}" for index in range(100)]
IMAGE_IDS = {
    "smoke": SMOKE_IMAGE_IDS,
    "validation": VALIDATION_IMAGE_IDS,
    "full": FULL_IMAGE_IDS,
}[RUN_MODE]
NUM_IMAGES = len(IMAGE_IDS)

MOUNT_GOOGLE_DRIVE = False
RUN_REFERENCE_DIFFPDHG = False
DIFFPDHG_METRICS_CSV = ""
DIFFPDHG_RECON_ROOT = ""
QUALITATIVE_IMAGE_IDS = ["L506_000", "L506_025", "L506_050", "L506_075"]

METHOD_CONFIGS = {
    "smoke": {
        "dps": {
            "num_inference_steps": 2,
            "scale": 10.0,
            "sirt_iterations": 2,
        },
        "red_diff": {
            "num_inference_steps": 2,
            "sigma": 1e-4,
            "loss_measurement_weight": 0.5,
            "loss_noise_weight": 10000.0,
            "learning_rate": 0.099,
            "sirt_iterations": 2,
        },
        "dmplug": {
            "num_inference_steps": 1,
            "optimization_iterations": 1,
            "epochs": 1,
            "learning_rate": 0.005,
            "initialization_seed": "BASE_SEED + image_index",
            "measurement_loss_type": "mse_line_integrals",
            "measurement_loss_weight": 1.0,
            "sirt_iterations": 2,
        },
        "dds": {
            "num_inference_steps": 2,
            "cg_inner": 1,
            "cg_eps": 1e-5,
            "gamma": 1.0,
            "eta": 0.85,
            "scheduler": "DDIMScheduler.from_config(checkpoint)",
            "sirt_iterations": 2,
        },
    },
    "validation": {
        "dps": {"num_inference_steps": 1000, "scale": 10.0, "sirt_iterations": 100},
        "red_diff": {
            "num_inference_steps": 200,
            "sigma": 1e-4,
            "loss_measurement_weight": 0.5,
            "loss_noise_weight": 10000.0,
            "learning_rate": 0.099,
            "sirt_iterations": 100,
        },
        "dmplug": {
            "num_inference_steps": 3,
            "optimization_iterations": 1000,
            "epochs": 1000,
            "learning_rate": 0.005,
            "initialization_seed": "BASE_SEED + image_index",
            "measurement_loss_type": "mse_line_integrals",
            "measurement_loss_weight": 1.0,
            "sirt_iterations": 100,
        },
        "dds": {
            "num_inference_steps": 100,
            "cg_inner": 5,
            "cg_eps": 1e-5,
            "gamma": 1.0,
            "eta": 0.85,
            "scheduler": "DDIMScheduler.from_config(checkpoint)",
            "sirt_iterations": 100,
        },
    },
    "full": {
        "dps": {"num_inference_steps": 1000, "scale": 10.0, "sirt_iterations": 100},
        "red_diff": {
            "num_inference_steps": 200,
            "sigma": 1e-4,
            "loss_measurement_weight": 0.5,
            "loss_noise_weight": 10000.0,
            "learning_rate": 0.099,
            "sirt_iterations": 100,
        },
        "dmplug": {
            "num_inference_steps": 3,
            "optimization_iterations": 1000,
            "epochs": 1000,
            "learning_rate": 0.005,
            "initialization_seed": "BASE_SEED + image_index",
            "measurement_loss_type": "mse_line_integrals",
            "measurement_loss_weight": 1.0,
            "sirt_iterations": 100,
        },
        "dds": {
            "num_inference_steps": 100,
            "cg_inner": 5,
            "cg_eps": 1e-5,
            "gamma": 1.0,
            "eta": 0.85,
            "scheduler": "DDIMScheduler.from_config(checkpoint)",
            "sirt_iterations": 100,
        },
    },
}

@dataclass(frozen=True)
class ExperimentConfig:
    RUN_MODE: str
    METHODS: List[str]
    DATA_ROOT: str
    VALIDATION_ROOT: str
    OUTPUT_ROOT: str
    MODEL_PATH_OR_HF_ID: str
    NUM_IMAGES: int
    IMAGE_IDS: List[str]
    NUM_ANGLES: int
    NUM_DETECTORS: int
    I0: float
    TRANSMITTANCE: float
    BASE_SEED: int
    SAVE_RECONSTRUCTIONS: bool
    RESUME: bool
    COMPUTE_LPIPS: bool
    ENABLE_POISSON_DMPLUG: bool
    ENABLE_LINEARIZED_TRACK: bool
    IMPLEMENTATION_REVISION: str
    METHOD_CONFIGS: Dict[str, Dict[str, Any]]
    CHECKPOINT_REVISION: str
    DM4CT_COMMIT: str
    DIFFPDHG_COMMIT: str

CONFIG = ExperimentConfig(
    RUN_MODE=RUN_MODE,
    METHODS=METHODS,
    DATA_ROOT=DATA_ROOT,
    VALIDATION_ROOT=VALIDATION_ROOT,
    OUTPUT_ROOT=OUTPUT_ROOT,
    MODEL_PATH_OR_HF_ID=MODEL_PATH_OR_HF_ID,
    NUM_IMAGES=NUM_IMAGES,
    IMAGE_IDS=IMAGE_IDS,
    NUM_ANGLES=NUM_ANGLES,
    NUM_DETECTORS=NUM_DETECTORS,
    I0=I0,
    TRANSMITTANCE=TRANSMITTANCE,
    BASE_SEED=BASE_SEED,
    SAVE_RECONSTRUCTIONS=SAVE_RECONSTRUCTIONS,
    RESUME=RESUME,
    COMPUTE_LPIPS=COMPUTE_LPIPS,
    ENABLE_POISSON_DMPLUG=ENABLE_POISSON_DMPLUG,
    ENABLE_LINEARIZED_TRACK=ENABLE_LINEARIZED_TRACK,
    IMPLEMENTATION_REVISION=IMPLEMENTATION_REVISION,
    METHOD_CONFIGS=METHOD_CONFIGS[RUN_MODE],
    CHECKPOINT_REVISION=CHECKPOINT_REVISION,
    DM4CT_COMMIT=DM4CT_COMMIT,
    DIFFPDHG_COMMIT=DIFFPDHG_COMMIT,
)

config_payload = asdict(CONFIG)
CONFIG_HASH = hashlib.sha256(
    json.dumps(config_payload, sort_keys=True, separators=(",", ":")).encode("utf-8")
).hexdigest()[:16]
OUTPUT_PATH = Path(CONFIG.OUTPUT_ROOT)
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

print(json.dumps({**config_payload, "CONFIG_HASH": CONFIG_HASH}, indent=2))


In [ ]:
def atomic_write_text(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(text, encoding="utf-8")
    os.replace(temporary, path)

def atomic_write_json(path, payload):
    atomic_write_text(path, json.dumps(payload, indent=2, sort_keys=True, default=str))

def atomic_write_csv(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)

LOG_PATH = OUTPUT_PATH / "run_log.txt"
LOGGER = logging.getLogger("dm4ct_extended")
LOGGER.setLevel(logging.INFO)
LOGGER.handlers.clear()
stream_handler = logging.StreamHandler(sys.stdout)
file_handler = logging.FileHandler(LOG_PATH, mode="a", encoding="utf-8")
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
stream_handler.setFormatter(formatter)
file_handler.setFormatter(formatter)
LOGGER.addHandler(stream_handler)
LOGGER.addHandler(file_handler)

REPOSITORY_HASHES = {
    "DM4CT": subprocess.check_output(
        ["git", "-C", str(DM4CT_DIR), "rev-parse", "HEAD"], text=True
    ).strip(),
    "DiffPDHG_reference": subprocess.check_output(
        ["git", "-C", str(DIFFPDHG_DIR), "rev-parse", "HEAD"], text=True
    ).strip(),
    "AAPM_pixel_diffusion_checkpoint": CHECKPOINT_REVISION,
}

ENVIRONMENT_MANIFEST = {
    "python_version": platform.python_version(),
    "pytorch_version": torch.__version__,
    "cuda_version": torch.version.cuda,
    "cudnn_version": torch.backends.cudnn.version(),
    "gpu_model": torch.cuda.get_device_name(0),
    "astra_version": getattr(astra, "__version__", "unknown"),
    "diffusers_version": importlib.metadata.version("diffusers"),
    "huggingface_hub_version": importlib.metadata.version("huggingface-hub"),
    "transformers_version": importlib.metadata.version("transformers"),
    "scikit_image_version": importlib.metadata.version("scikit-image"),
    "repository_commit_hashes": REPOSITORY_HASHES,
    "diffusion_checkpoint_identifier": CONFIG.MODEL_PATH_OR_HF_ID,
    "diffusion_checkpoint_revision": CHECKPOINT_REVISION,
    "determinism_note": (
        "Python/NumPy/PyTorch seeds and deterministic cuDNN are set. "
        "ASTRA/CUDA reductions may not be bitwise deterministic across hardware or drivers."
    ),
}
atomic_write_json(OUTPUT_PATH / "experiment_config.json", {**config_payload, "CONFIG_HASH": CONFIG_HASH})
atomic_write_json(OUTPUT_PATH / "environment_manifest.json", ENVIRONMENT_MANIFEST)
print(json.dumps(ENVIRONMENT_MANIFEST, indent=2))


## 6. Dataset and checkpoint loading

Smoke mode uses DM4CT's repository sample. Validation and full modes fail immediately unless every explicitly listed normalized TIFF is present. No fallback slice substitution is allowed. The single checkpoint is loaded once, and the same UNet object is passed to every pipeline.


In [ ]:
def resolve_image_records(config):
    if config.RUN_MODE == "smoke":
        root = DM4CT_DIR / "lodochallenge"
    elif config.RUN_MODE == "validation":
        root = Path(config.VALIDATION_ROOT)
    else:
        root = Path(config.DATA_ROOT)
    if not root.exists():
        raise FileNotFoundError(f"Dataset root does not exist for {config.RUN_MODE}: {root}")

    records = []
    for image_id in config.IMAGE_IDS:
        candidates = [root / f"{image_id}.tif", root / f"{image_id}.tiff"]
        matches = [path…14461 tokens truncated…sk.operator.num_angles=80",
        "inverse_task.operator.num_detectors=512",
        "inverse_task.operator.measurement_mode=poisson",
        "model.model_config.model_id=jiayangshi/lodochallenge_pixel_diffusion",
        f"data.image_root_path={(DIFFPDHG_DIR / 'demo-samples' / 'ct_l067_subset_tiff').as_posix()}",
        "data.start_idx=1",
        "data.end_idx=2",
    ]
    subprocess.run(reference_command, cwd=DIFFPDHG_DIR, check=True)
else:
    print("Supplied DiffPDHG raw-count reference launcher retained but disabled.")


## 18. Aggregate metrics and confidence intervals

Summary statistics use successful final reconstructions. Bootstrap resampling uses the common successful image IDs across methods, preserving pairing. Optional DiffPDHG differences are emitted only when an external matching-ID CSV is configured, and compatibility is labelled explicitly.


In [ ]:
SUMMARY_PATH = OUTPUT_PATH / "metrics_summary.csv"
PAIRWISE_PATH = OUTPUT_PATH / "paired_differences_vs_diffpdhg.csv"

def bootstrap_mean_ci(values, seed=99, draws=2000):
    values = np.asarray(values, dtype=np.float64)
    if values.size == 0:
        return np.nan, np.nan
    generator = np.random.default_rng(seed)
    samples = generator.choice(values, size=(draws, values.size), replace=True).mean(axis=1)
    return tuple(np.quantile(samples, [0.025, 0.975]))

if METRICS_PATH.exists():
    metrics_frame = pd.read_csv(METRICS_PATH, keep_default_na=False)
    scoped = metrics_frame[
        (metrics_frame["config_hash"] == CONFIG_HASH)
        & (metrics_frame["image_id"].isin(CONFIG.IMAGE_IDS))
    ].copy()
    for column in [
        "psnr", "ssim", "lpips", "runtime_seconds", "nfe",
        "forward_operator_calls", "adjoint_operator_calls",
        "cg_iterations", "optimizer_iterations",
    ]:
        scoped[column] = pd.to_numeric(scoped[column], errors="coerce")
    successful = scoped[scoped["status"] == "success"].copy()
    method_id_sets = [
        set(group["image_id"]) for _, group in successful.groupby("method")
    ]
    common_ids = set.intersection(*method_id_sets) if method_id_sets else set()
    summary_rows = []
    for method, group in scoped.groupby("method"):
        success_group = group[
            (group["status"] == "success") & (group["image_id"].isin(common_ids))
        ]
        row = {
            "method": method,
            "measurement_model": (
                success_group["measurement_model"].iloc[0] if len(success_group) else ""
            ),
            "successful_reconstructions": int((group["status"] == "success").sum()),
            "failures": int((group["status"] != "success").sum()),
            "paired_successful_images": len(success_group),
        }
        for metric in [
            "psnr", "ssim", "lpips", "runtime_seconds", "nfe",
            "forward_operator_calls", "adjoint_operator_calls",
            "cg_iterations", "optimizer_iterations",
        ]:
            values = success_group[metric].dropna().to_numpy(dtype=np.float64)
            low, high = bootstrap_mean_ci(
                values, seed=CONFIG.BASE_SEED + sum(ord(c) for c in method + metric)
            )
            row.update(
                {
                    f"{metric}_mean": np.mean(values) if values.size else np.nan,
                    f"{metric}_std": np.std(values, ddof=1) if values.size > 1 else 0.0 if values.size else np.nan,
                    f"{metric}_median": np.median(values) if values.size else np.nan,
                    f"{metric}_iqr": (
                        np.quantile(values, 0.75) - np.quantile(values, 0.25)
                        if values.size else np.nan
                    ),
                    f"{metric}_paired_bootstrap_ci95_low": low,
                    f"{metric}_paired_bootstrap_ci95_high": high,
                }
            )
        summary_rows.append(row)
    summary_frame = pd.DataFrame(summary_rows)
else:
    scoped = pd.DataFrame(columns=ALL_METRIC_COLUMNS)
    summary_frame = pd.DataFrame()
atomic_write_csv(summary_frame, SUMMARY_PATH)
display(summary_frame)

pairwise_columns = [
    "image_id", "method", "metric", "diffusion_value", "diffpdhg_value",
    "difference_method_minus_diffpdhg", "measurement_compatible", "compatibility_note",
]
pairwise_rows = []
if DIFFPDHG_METRICS_CSV and Path(DIFFPDHG_METRICS_CSV).exists():
    diffpdhg = pd.read_csv(DIFFPDHG_METRICS_CSV)
    required = {"image_id", "psnr", "ssim"}
    if not required.issubset(diffpdhg.columns):
        raise ValueError(f"DiffPDHG CSV must contain {sorted(required)}")
    for _, row in scoped[scoped["status"] == "success"].iterrows():
        reference = diffpdhg[diffpdhg["image_id"] == row["image_id"]]
        if reference.empty:
            continue
        for metric in ["psnr", "ssim"]:
            pairwise_rows.append(
                {
                    "image_id": row["image_id"],
                    "method": row["method"],
                    "metric": metric,
                    "diffusion_value": float(row[metric]),
                    "diffpdhg_value": float(reference.iloc[0][metric]),
                    "difference_method_minus_diffpdhg": float(row[metric])
                    - float(reference.iloc[0][metric]),
                    "measurement_compatible": False,
                    "compatibility_note": (
                        "Supplied DiffPDHG is custom-projector raw Poisson; DM4CT primary paths are "
                        "ASTRA log-linearized. Difference is descriptive, not likelihood-matched."
                    ),
                }
            )
atomic_write_csv(pd.DataFrame(pairwise_rows, columns=pairwise_columns), PAIRWISE_PATH)


In [ ]:
strict_rows = []
extended_rows = []
if not summary_frame.empty:
    for _, row in summary_frame.iterrows():
        display_name = {
            "dps": "DPS",
            "red_diff": "RED-diff",
            "dmplug_linearized": "DMPlug (log-linearized)",
            "dmplug_poisson": "DMPlug (Poisson counts)",
            "dds": "DDS (log-linearized)",
        }.get(row["method"], row["method"])
        table_row = {
            "Method": display_name,
            "Diffusion checkpoint": f"{CONFIG.MODEL_PATH_OR_HF_ID}@{CHECKPOINT_REVISION[:8]}",
            "Measurement representation": row["measurement_model"],
            "Data-consistency model": {
                "dps": "linear ASTRA residual guidance",
                "red_diff": "linear ASTRA squared loss",
                "dmplug_linearized": "linear ASTRA MSE through generator",
                "dmplug_poisson": "Beer-Lambert Poisson NLL through generator",
                "dds": "linear ASTRA conjugate gradient",
            }.get(row["method"], ""),
            "NFE": row.get("nfe_mean", np.nan),
            "Runtime": row.get("runtime_seconds_mean", np.nan),
            "PSNR": row.get("psnr_mean", np.nan),
            "SSIM": row.get("ssim_mean", np.nan),
        }
        extended_rows.append(table_row)
        if row["measurement_model"] == "log_linearized_poisson":
            strict_rows.append(table_row)

strict_table = pd.DataFrame(strict_rows)
extended_table = pd.DataFrame(extended_rows)
atomic_write_csv(strict_table, OUTPUT_PATH / "strict_common_measurement_table.csv")
atomic_write_csv(extended_table, OUTPUT_PATH / "extended_diffusion_prior_table.csv")
print("Strict common-measurement table")
display(strict_table)
print("Extended diffusion-prior table")
display(extended_table)


## 19. Qualitative comparison

IDs are fixed above before results are inspected. All ground truth and reconstruction panels use the same `[-1, 1]` grayscale window. The sinogram column uses one per-slice sinogram scale because it is a different physical domain. Metrics under each reconstruction are final-iterate values.


In [ ]:
def load_saved_reconstruction(image_id, method):
    npz_path, _, _ = reconstruction_paths(image_id, method)
    if not npz_path.exists():
        return None
    with np.load(npz_path, allow_pickle=False) as saved:
        if str(saved["config_hash"].item()) != CONFIG_HASH:
            return None
        return np.asarray(saved["reconstruction"], dtype=np.float32)

def load_external_diffpdhg_reconstruction(image_id):
    if not DIFFPDHG_RECON_ROOT:
        return None
    root = Path(DIFFPDHG_RECON_ROOT)
    candidates = [
        root / f"{image_id}.npy",
        root / f"{image_id}.npz",
        root / f"{image_id}.tif",
        root / f"{image_id}.tiff",
    ]
    for path in candidates:
        if not path.exists():
            continue
        if path.suffix == ".npy":
            array = np.load(path, allow_pickle=False)
        elif path.suffix == ".npz":
            with np.load(path, allow_pickle=False) as saved:
                key = "reconstruction" if "reconstruction" in saved.files else saved.files[0]
                array = saved[key]
        else:
            array = imread(path)
        array = np.asarray(array, dtype=np.float32).squeeze()
        if array.shape != (512, 512) or not np.isfinite(array).all():
            raise ValueError(f"Invalid external DiffPDHG reconstruction: {path}")
        return array
    return None

available_qualitative_ids = [
    image_id
    for image_id in QUALITATIVE_IMAGE_IDS
    if image_id in MEASUREMENT_BUNDLES
]
if CONFIG.RUN_MODE == "smoke" and not available_qualitative_ids:
    available_qualitative_ids = CONFIG.IMAGE_IDS[:1]

panel_methods = expanded_method_names()
include_diffpdhg = any(
    load_external_diffpdhg_reconstruction(image_id) is not None
    for image_id in available_qualitative_ids
)
if available_qualitative_ids and METRICS_PATH.exists():
    panel_frame = pd.read_csv(METRICS_PATH, keep_default_na=False)
    columns = (
        ["Ground truth", "Log sinogram"]
        + panel_methods
        + (["DiffPDHG"] if include_diffpdhg else [])
    )
    figure, axes = plt.subplots(
        len(available_qualitative_ids),
        len(columns),
        figsize=(3.0 * len(columns), 3.4 * len(available_qualitative_ids)),
        squeeze=False,
    )
    for row_index, image_id in enumerate(available_qualitative_ids):
        bundle = MEASUREMENT_BUNDLES[image_id]
        ground_truth = bundle["ground_truth"].numpy().squeeze()
        axes[row_index, 0].imshow(ground_truth, cmap="gray", vmin=-1, vmax=1)
        axes[row_index, 0].set_title("Ground truth")
        axes[row_index, 0].set_xlabel(image_id)
        sinogram = bundle["log_line_integrals"].numpy().squeeze()
        axes[row_index, 1].imshow(sinogram, cmap="gray", aspect="auto")
        axes[row_index, 1].set_title("Log sinogram")
        axes[row_index, 1].set_xlabel(
            f"{CONFIG.NUM_ANGLES} views x {CONFIG.NUM_DETECTORS} bins"
        )
        for column_index, method in enumerate(panel_methods, start=2):
            reconstruction = load_saved_reconstruction(image_id, method)
            axes[row_index, column_index].set_title(
                {
                    "dps": "DPS",
                    "red_diff": "RED-diff",
                    "dmplug_linearized": "DMPlug",
                    "dmplug_poisson": "DMPlug Poisson",
                    "dds": "DDS",
                }.get(method, method)
            )
            if reconstruction is None:
                axes[row_index, column_index].text(
                    0.5, 0.5, "not available", ha="center", va="center"
                )
            else:
                axes[row_index, column_index].imshow(
                    reconstruction, cmap="gray", vmin=-1, vmax=1
                )
                metric_row = panel_frame[
                    (panel_frame["config_hash"] == CONFIG_HASH)
                    & (panel_frame["image_id"] == image_id)
                    & (panel_frame["method"] == method)
                    & (panel_frame["status"] == "success")
                ]
                if not metric_row.empty:
                    metric_row = metric_row.iloc[0]
                    axes[row_index, column_index].set_xlabel(
                        f"PSNR {float(metric_row['psnr']):.2f} | "
                        f"SSIM {float(metric_row['ssim']):.3f}\n"
                        f"{float(metric_row['runtime_seconds']):.1f}s | "
                        f"NFE {int(float(metric_row['nfe']))}"
                    )
        if include_diffpdhg:
            column_index = len(columns) - 1
            axes[row_index, column_index].set_title("DiffPDHG")
            reconstruction = load_external_diffpdhg_reconstruction(image_id)
            if reconstruction is None:
                axes[row_index, column_index].text(
                    0.5, 0.5, "not available", ha="center", va="center"
                )
            else:
                axes[row_index, column_index].imshow(
                    reconstruction, cmap="gray", vmin=-1, vmax=1
                )
                if DIFFPDHG_METRICS_CSV and Path(DIFFPDHG_METRICS_CSV).exists():
                    reference_metrics = pd.read_csv(DIFFPDHG_METRICS_CSV)
                    reference_row = reference_metrics[
                        reference_metrics["image_id"] == image_id
                    ]
                    if not reference_row.empty:
                        reference_row = reference_row.iloc[0]
                        runtime = pd.to_numeric(
                            pd.Series([reference_row.get("runtime_seconds", np.nan)]),
                            errors="coerce",
                        ).iloc[0]
                        nfe = pd.to_numeric(
                            pd.Series([reference_row.get("nfe", np.nan)]),
                            errors="coerce",
                        ).iloc[0]
                        axes[row_index, column_index].set_xlabel(
                            f"PSNR {float(reference_row['psnr']):.2f} | "
                            f"SSIM {float(reference_row['ssim']):.3f}\n"
                            f"{float(runtime):.1f}s | NFE {int(float(nfe))}"
                            if np.isfinite(float(runtime)) and np.isfinite(float(nfe))
                            else (
                                f"PSNR {float(reference_row['psnr']):.2f} | "
                                f"SSIM {float(reference_row['ssim']):.3f}"
                            )
                        )
        for axis in axes[row_index]:
            axis.set_xticks([])
            axis.set_yticks([])
    figure.tight_layout()
    panel_png = OUTPUT_PATH / "qualitative_comparison.png"
    panel_pdf = OUTPUT_PATH / "qualitative_comparison.pdf"
    figure.savefig(panel_png, dpi=180, bbox_inches="tight")
    figure.savefig(panel_pdf, bbox_inches="tight")
    plt.show()
    print(panel_png, panel_pdf)
else:
    print("No completed primary reconstructions are available for a qualitative panel.")


## 20. Export and run summary

The output root contains the experiment/config manifests, shared measurements, atomic per-image reconstructions, diagnostics, all requested CSVs, logs, smoke output, and qualitative panels. Copy or point `OUTPUT_ROOT` to Google Drive for persistent full runs.


In [ ]:
required_outputs = [
    "metrics_per_image.csv",
    "metrics_summary.csv",
    "paired_differences_vs_diffpdhg.csv",
    "experiment_config.json",
    "environment_manifest.json",
    "validation_results.csv",
    "run_log.txt",
]
output_status = {
    name: {
        "exists": (OUTPUT_PATH / name).exists(),
        "path": str(OUTPUT_PATH / name),
    }
    for name in required_outputs
}
run_summary = {
    "run_mode": CONFIG.RUN_MODE,
    "config_hash": CONFIG_HASH,
    "image_ids": CONFIG.IMAGE_IDS,
    "methods": expanded_method_names(),
    "outputs": output_status,
    "dds_same_measurement_model_as_supplied_diffpdhg": False,
    "dds_measurement_statement": (
        "DDS is evaluated on the shared DM4CT log-linearized ASTRA sinogram. "
        "The supplied DiffPDHG reference uses raw photon counts with a custom non-ASTRA operator."
    ),
}
atomic_write_json(OUTPUT_PATH / "run_summary.json", run_summary)
print(json.dumps(run_summary, indent=2))


## Changes relative to the reference notebook

- **Added imports:** ASTRA, DM4CT pipeline/conditioning classes, diffusers schedulers, tifffile, pandas, scikit-image metrics, optional LPIPS, plotting, hashing, dataclasses, logging, traceback, and reproducibility utilities.
- **Added repository files:** no source files are copied into this notebook; it checks out official DM4CT at `49b3e59...` and retains the supplied DiffPDHG repository at `35db526...`. The only local DDS code is a diagnostic subclass of the official pinned pipeline loop.
- **Changed package versions:** ASTRA is changed from DM4CT's 2.3.0 environment to 2.5.0 for current Colab/PyPI CUDA-wheel support. DM4CT's diffusers 0.32.2 stack is retained. Colab supplies PyTorch/CUDA and their actual versions are manifested.
- **Added method adapters:** unified `run_method` plus `run_dps`, `run_red_diff`, `run_dmplug`, and `run_dds`; optional `dmplug_poisson`; actual NFE/operator counters; DDS CG diagnostics; DMPlug loss recording.
- **Added configuration entries:** all required run/data/model/geometry/noise/seed/output/resume/LPIPS/measurement-track fields, three run modes, fixed test and validation IDs, and frozen method settings.
- **Added evaluation outputs:** cached measurement bundles, `metrics_per_image.csv`, aggregate/bootstrap summaries, optional DiffPDHG differences, strict and extended tables, validation records, manifests, run log, atomic reconstructions/diagnostics, and PNG/PDF qualitative panels.
- **Necessary DPS/RED-diff modifications:** none to their DM4CT algorithms. They are wrapped only for common inputs, exact counters, timing, final-residual diagnostics, and shortened smoke settings.
- **Reference DiffPDHG cell:** retained behind `RUN_REFERENCE_DIFFPDHG`; its original raw-count/non-ASTRA/`I0=10000` contract remains separate and disabled by default.
